# COMP 469 — Chapter 3 Search, Self-Contained

This notebook contains the Chapter 3 search subset needed for classroom demos. It does not require the full `aima-python` repository.

Included: `Problem`, `Node`, Romania route problem, BFS, DFS, UCS, DLS, IDS, greedy best-first, A*, and RBFS.

Also included: an animated Romania map visualization for BFS, DFS, UCS, greedy best-first, and A*.


In [ ]:
"""
COMP 469 / AIMA Chapter 3 Search - Self-Contained Classroom Subset

This file is intentionally standalone: it uses only the Python standard library.

Included:
- Problem
- Node
- SimpleProblemSolvingAgent
- Graph and Romania map
- breadth_first_tree_search
- depth_first_tree_search
- breadth_first_graph_search
- depth_first_graph_search
- uniform_cost_search
- depth_limited_search
- iterative_deepening_search
- best_first_graph_search
- astar_search
- recursive_best_first_search

Run:
    python aima_ch3_search_standalone.py
"""

from __future__ import annotations

from collections import deque
from dataclasses import dataclass
import heapq
import itertools
import math
from typing import Any, Callable, Dict, Iterable, List, Optional, Tuple


class Problem:
    """Abstract search problem.

    To define a concrete problem, subclass this and implement actions() and result().
    Override goal_test(), path_cost(), and h() when needed.
    """

    def __init__(self, initial: Any, goal: Optional[Any] = None):
        self.initial = initial
        self.goal = goal

    def actions(self, state: Any) -> Iterable[Any]:
        """Return actions available in the given state."""
        raise NotImplementedError

    def result(self, state: Any, action: Any) -> Any:
        """Return the state that results from doing action in state."""
        raise NotImplementedError

    def goal_test(self, state: Any) -> bool:
        """Return True if state is a goal state."""
        if isinstance(self.goal, (list, tuple, set)):
            return state in self.goal
        return state == self.goal

    def path_cost(self, cost_so_far: float, state1: Any, action: Any, state2: Any) -> float:
        """Return total path cost after moving from state1 to state2 by action."""
        return cost_so_far + 1

    def h(self, node: "Node") -> float:
        """Heuristic estimate from node to goal. Default is uninformed."""
        return 0


@dataclass
class Node:
    """A node in a search tree."""

    state: Any
    parent: Optional["Node"] = None
    action: Optional[Any] = None
    path_cost: float = 0
    depth: int = 0

    def __post_init__(self):
        if self.parent is not None:
            self.depth = self.parent.depth + 1

    def __lt__(self, other: "Node") -> bool:
        return self.path_cost < other.path_cost

    def child_node(self, problem: Problem, action: Any) -> "Node":
        next_state = problem.result(self.state, action)
        next_cost = problem.path_cost(self.path_cost, self.state, action, next_state)
        return Node(next_state, self, action, next_cost)

    def expand(self, problem: Problem) -> List["Node"]:
        return [self.child_node(problem, action) for action in problem.actions(self.state)]

    def solution(self) -> List[Any]:
        """Return actions from root to this node."""
        return [node.action for node in self.path()[1:]]

    def path(self) -> List["Node"]:
        """Return nodes from root to this node."""
        node = self
        result = []
        while node is not None:
            result.append(node)
            node = node.parent
        return list(reversed(result))

    def path_states(self) -> List[Any]:
        return [node.state for node in self.path()]


class PriorityQueue:
    """Small priority queue wrapper for search frontiers."""

    def __init__(self, f: Callable[[Node], float]):
        self.f = f
        self.heap: List[Tuple[float, int, Node]] = []
        self.counter = itertools.count()

    def append(self, node: Node) -> None:
        heapq.heappush(self.heap, (self.f(node), next(self.counter), node))

    def pop(self) -> Node:
        return heapq.heappop(self.heap)[2]

    def __len__(self) -> int:
        return len(self.heap)


class Graph:
    """Weighted graph.

    graph_dict format:
        {
            'A': {'B': 5, 'C': 2},
            'B': {'D': 4}
        }
    """

    def __init__(self, graph_dict: Optional[Dict[Any, Dict[Any, float]]] = None, directed: bool = True):
        self.graph_dict: Dict[Any, Dict[Any, float]] = graph_dict or {}
        self.directed = directed
        if not directed:
            self.make_undirected()

    def make_undirected(self) -> None:
        for a in list(self.graph_dict.keys()):
            for b, distance in list(self.graph_dict[a].items()):
                self.connect(b, a, distance)

    def connect(self, a: Any, b: Any, distance: float = 1) -> None:
        self.graph_dict.setdefault(a, {})[b] = distance
        if not self.directed:
            self.graph_dict.setdefault(b, {})[a] = distance

    def get(self, a: Any, b: Optional[Any] = None):
        links = self.graph_dict.setdefault(a, {})
        if b is None:
            return links
        return links.get(b, math.inf)

    def nodes(self) -> List[Any]:
        result = set(self.graph_dict.keys())
        for neighbors in self.graph_dict.values():
            result.update(neighbors.keys())
        return sorted(result)


class GraphProblem(Problem):
    """Route-finding problem on a weighted graph."""

    def __init__(self, initial: Any, goal: Any, graph: Graph, heuristic: Optional[Dict[Any, float]] = None):
        super().__init__(initial, goal)
        self.graph = graph
        self.heuristic = heuristic or {}

    def actions(self, state: Any) -> Iterable[Any]:
        return self.graph.get(state).keys()

    def result(self, state: Any, action: Any) -> Any:
        return action

    def path_cost(self, cost_so_far: float, state1: Any, action: Any, state2: Any) -> float:
        return cost_so_far + self.graph.get(state1, state2)

    def h(self, node: Node) -> float:
        return self.heuristic.get(node.state, 0)


romania_map = Graph(
    {
        "Arad": {"Zerind": 75, "Sibiu": 140, "Timisoara": 118},
        "Bucharest": {"Urziceni": 85, "Pitesti": 101, "Giurgiu": 90, "Fagaras": 211},
        "Craiova": {"Drobeta": 120, "Rimnicu Vilcea": 146, "Pitesti": 138},
        "Drobeta": {"Mehadia": 75},
        "Eforie": {"Hirsova": 86},
        "Fagaras": {"Sibiu": 99},
        "Hirsova": {"Urziceni": 98},
        "Iasi": {"Vaslui": 92, "Neamt": 87},
        "Lugoj": {"Timisoara": 111, "Mehadia": 70},
        "Oradea": {"Zerind": 71, "Sibiu": 151},
        "Pitesti": {"Rimnicu Vilcea": 97},
        "Rimnicu Vilcea": {"Sibiu": 80},
        "Urziceni": {"Vaslui": 142},
    },
    directed=False,
)

# Straight-line distance to Bucharest, commonly used for A* on the Romania map.
romania_sld_to_bucharest = {
    "Arad": 366,
    "Bucharest": 0,
    "Craiova": 160,
    "Drobeta": 242,
    "Eforie": 161,
    "Fagaras": 176,
    "Giurgiu": 77,
    "Hirsova": 151,
    "Iasi": 226,
    "Lugoj": 244,
    "Mehadia": 241,
    "Neamt": 234,
    "Oradea": 380,
    "Pitesti": 100,
    "Rimnicu Vilcea": 193,
    "Sibiu": 253,
    "Timisoara": 329,
    "Urziceni": 80,
    "Vaslui": 199,
    "Zerind": 374,
}


def breadth_first_tree_search(problem: Problem) -> Optional[Node]:
    frontier = deque([Node(problem.initial)])
    while frontier:
        node = frontier.popleft()
        if problem.goal_test(node.state):
            return node
        frontier.extend(node.expand(problem))
    return None


def depth_first_tree_search(problem: Problem, limit: int = 50) -> Optional[Node]:
    frontier = [Node(problem.initial)]
    while frontier:
        node = frontier.pop()
        if problem.goal_test(node.state):
            return node
        if node.depth < limit:
            frontier.extend(reversed(node.expand(problem)))
    return None


def breadth_first_graph_search(problem: Problem) -> Optional[Node]:
    node = Node(problem.initial)
    if problem.goal_test(node.state):
        return node

    frontier = deque([node])
    reached = {problem.initial}

    while frontier:
        node = frontier.popleft()
        for child in node.expand(problem):
            if child.state not in reached:
                if problem.goal_test(child.state):
                    return child
                reached.add(child.state)
                frontier.append(child)
    return None


def depth_first_graph_search(problem: Problem, limit: int = 50) -> Optional[Node]:
    frontier = [Node(problem.initial)]
    reached = set()

    while frontier:
        node = frontier.pop()
        if problem.goal_test(node.state):
            return node
        if node.state not in reached and node.depth < limit:
            reached.add(node.state)
            frontier.extend(reversed(node.expand(problem)))
    return None


def best_first_graph_search(problem: Problem, f: Callable[[Node], float]) -> Optional[Node]:
    node = Node(problem.initial)
    frontier = PriorityQueue(f)
    frontier.append(node)

    best_score = {node.state: f(node)}

    while frontier:
        node = frontier.pop()

        if problem.goal_test(node.state):
            return node

        for child in node.expand(problem):
            score = f(child)
            if child.state not in best_score or score < best_score[child.state]:
                best_score[child.state] = score
                frontier.append(child)

    return None


def uniform_cost_search(problem: Problem) -> Optional[Node]:
    return best_first_graph_search(problem, lambda node: node.path_cost)


def astar_search(problem: Problem, h: Optional[Callable[[Node], float]] = None) -> Optional[Node]:
    heuristic = h or problem.h
    return best_first_graph_search(problem, lambda node: node.path_cost + heuristic(node))


def depth_limited_search(problem: Problem, limit: int = 50):
    """Return a solution Node, None for failure, or the string 'cutoff'."""

    def recursive_dls(node: Node, problem: Problem, limit_remaining: int):
        if problem.goal_test(node.state):
            return node
        if limit_remaining == 0:
            return "cutoff"

        cutoff_occurred = False
        for child in node.expand(problem):
            # Avoid trivial cycles on graph problems.
            if child.state in node.path_states():
                continue

            result = recursive_dls(child, problem, limit_remaining - 1)
            if result == "cutoff":
                cutoff_occurred = True
            elif result is not None:
                return result

        return "cutoff" if cutoff_occurred else None

    return recursive_dls(Node(problem.initial), problem, limit)


def iterative_deepening_search(problem: Problem, max_depth: int = 50) -> Optional[Node]:
    for depth in range(max_depth + 1):
        result = depth_limited_search(problem, depth)
        if result != "cutoff":
            return result
    return None


def recursive_best_first_search(problem: Problem, h: Optional[Callable[[Node], float]] = None) -> Optional[Node]:
    heuristic = h or problem.h

    def rbfs(node: Node, f_limit: float) -> Tuple[Optional[Node], float]:
        if problem.goal_test(node.state):
            return node, 0

        successors = []
        current_path_states = set(node.path_states())

        for child in node.expand(problem):
            if child.state not in current_path_states:
                child.f = max(child.path_cost + heuristic(child), getattr(node, "f", 0))
                successors.append(child)

        if not successors:
            return None, math.inf

        while True:
            successors.sort(key=lambda n: n.f)
            best = successors[0]

            if best.f > f_limit:
                return None, best.f

            alternative = successors[1].f if len(successors) > 1 else math.inf
            result, best.f = rbfs(best, min(f_limit, alternative))

            if result is not None:
                return result, best.f

    start = Node(problem.initial)
    start.f = heuristic(start)
    result, _ = rbfs(start, math.inf)
    return result


class SimpleProblemSolvingAgent:
    """A tiny configurable simple problem-solving agent.

    This is a classroom-friendly version:
    - The user provides a problem_factory that creates a Problem from a state and goal.
    - The user provides a search algorithm.
    - The agent computes a sequence of actions, then returns one action per call.
    """

    def __init__(
        self,
        initial_state: Any,
        goal: Any,
        problem_factory: Callable[[Any, Any], Problem],
        search_algorithm: Callable[[Problem], Optional[Node]] = breadth_first_graph_search,
    ):
        self.state = initial_state
        self.goal = goal
        self.problem_factory = problem_factory
        self.search_algorithm = search_algorithm
        self.action_sequence: List[Any] = []

    def update_state(self, percept: Any) -> Any:
        self.state = percept
        return self.state

    def formulate_goal(self, state: Any) -> Any:
        return self.goal

    def formulate_problem(self, state: Any, goal: Any) -> Problem:
        return self.problem_factory(state, goal)

    def __call__(self, percept: Any) -> Optional[Any]:
        state = self.update_state(percept)

        if not self.action_sequence:
            goal = self.formulate_goal(state)
            problem = self.formulate_problem(state, goal)
            solution_node = self.search_algorithm(problem)
            self.action_sequence = solution_node.solution() if solution_node else []

        if self.action_sequence:
            return self.action_sequence.pop(0)

        return None


def print_solution(name: str, node: Optional[Node]) -> None:
    if node is None:
        print(f"{name}: no solution")
        return

    print(f"{name}:")
    print(f"  Path: {' -> '.join(map(str, node.path_states()))}")
    print(f"  Actions: {node.solution()}")
    print(f"  Cost: {node.path_cost}")
    print(f"  Depth: {node.depth}")
    print()


def demo_romania() -> None:
    problem = GraphProblem("Arad", "Bucharest", romania_map, romania_sld_to_bucharest)

    algorithms = [
        ("Breadth-first graph search", breadth_first_graph_search),
        ("Depth-first graph search", depth_first_graph_search),
        ("Uniform-cost search", uniform_cost_search),
        ("Depth-limited search, limit=3", lambda p: depth_limited_search(p, limit=3)),
        ("Iterative-deepening search", iterative_deepening_search),
        ("Greedy best-first search", lambda p: best_first_graph_search(p, p.h)),
        ("A* search", astar_search),
        ("Recursive best-first search", recursive_best_first_search),
    ]

    for name, algorithm in algorithms:
        print_solution(name, algorithm(problem))




## Demo: Romania route problem


In [ ]:
problem = GraphProblem("Arad", "Bucharest", romania_map, romania_sld_to_bucharest)

algorithms = [
    ("Breadth-first graph search", breadth_first_graph_search),
    ("Depth-first graph search", depth_first_graph_search),
    ("Uniform-cost search", uniform_cost_search),
    ("Depth-limited search, limit=3", lambda p: depth_limited_search(p, limit=3)),
    ("Iterative-deepening search", iterative_deepening_search),
    ("Greedy best-first search", lambda p: best_first_graph_search(p, p.h)),
    ("A* search", astar_search),
    ("Recursive best-first search", recursive_best_first_search),
]

for name, algorithm in algorithms:
    print_solution(name, algorithm(problem))


## Visual demo: animated search on the Romania map

Run the next two cells to see the frontier, expanded states, current node, and final path. Use the algorithm menu in the output to compare how BFS, DFS, UCS, greedy best-first, and A* move through the same state space.


In [ ]:
import json
import uuid
from IPython.display import HTML

# Approximate screen coordinates for the AIMA Romania road map.
romania_locations = {
    "Arad": (92, 178),
    "Zerind": (108, 120),
    "Oradea": (138, 66),
    "Sibiu": (215, 170),
    "Timisoara": (88, 258),
    "Lugoj": (168, 304),
    "Mehadia": (172, 352),
    "Drobeta": (166, 400),
    "Craiova": (260, 414),
    "Rimnicu Vilcea": (247, 250),
    "Fagaras": (322, 178),
    "Pitesti": (338, 316),
    "Bucharest": (430, 372),
    "Giurgiu": (404, 438),
    "Urziceni": (502, 338),
    "Hirsova": (578, 330),
    "Eforie": (612, 388),
    "Vaslui": (542, 224),
    "Iasi": (498, 126),
    "Neamt": (434, 78),
}

romania_label_offsets = {
    "Arad": (-12, -12, "end"),
    "Zerind": (-10, -8, "end"),
    "Oradea": (-8, -12, "end"),
    "Sibiu": (0, -17, "middle"),
    "Timisoara": (-12, 2, "end"),
    "Lugoj": (10, -10, "start"),
    "Mehadia": (12, 0, "start"),
    "Drobeta": (-10, 16, "end"),
    "Craiova": (0, 20, "middle"),
    "Rimnicu Vilcea": (0, -18, "middle"),
    "Fagaras": (0, -17, "middle"),
    "Pitesti": (8, -12, "start"),
    "Bucharest": (12, 14, "start"),
    "Giurgiu": (0, 22, "middle"),
    "Urziceni": (10, -10, "start"),
    "Hirsova": (12, -8, "start"),
    "Eforie": (12, 4, "start"),
    "Vaslui": (12, -8, "start"),
    "Iasi": (12, -8, "start"),
    "Neamt": (0, -16, "middle"),
}


def _romania_edges_for_display(graph):
    edges = []
    seen = set()
    for a in graph.nodes():
        for b, distance in graph.get(a).items():
            key = tuple(sorted((a, b)))
            if key in seen:
                continue
            seen.add(key)
            edges.append({"from": a, "to": b, "distance": distance})
    return edges


def _frontier_entries(frontier_nodes, problem, score_fn=None):
    entries = []
    for node in frontier_nodes:
        entry = {
            "state": node.state,
            "g": node.path_cost,
            "depth": node.depth,
            "h": problem.h(node),
        }
        if score_fn is not None:
            entry["score"] = score_fn(node)
        entries.append(entry)
    return entries


def _make_trace_frame(
    *,
    step,
    message,
    problem,
    current=None,
    frontier_nodes=None,
    expanded=None,
    reached=None,
    solution=None,
    score_label="",
    score_fn=None,
):
    frontier_nodes = frontier_nodes or []
    current_path = current.path_states() if current is not None else []
    solution_path = solution.path_states() if solution is not None else []
    return {
        "step": step,
        "message": message,
        "current": current.state if current is not None else None,
        "current_cost": current.path_cost if current is not None else 0,
        "current_depth": current.depth if current is not None else 0,
        "path": current_path,
        "solution_path": solution_path,
        "solution_cost": solution.path_cost if solution is not None else None,
        "frontier": _frontier_entries(frontier_nodes, problem, score_fn),
        "expanded": list(expanded or []),
        "reached": sorted(reached or []),
        "score_label": score_label,
    }


def _trace_queue_search(problem, mode="bfs", limit=50):
    is_bfs = mode == "bfs"
    title = "Breadth-first graph search" if is_bfs else "Depth-first graph search"
    frontier = deque([Node(problem.initial)]) if is_bfs else [Node(problem.initial)]
    reached = {problem.initial} if is_bfs else set()
    expanded = []
    frames = []

    def frontier_snapshot():
        nodes = list(frontier)
        return nodes if is_bfs else list(reversed(nodes))

    def add_frame(message, current=None, solution=None):
        frames.append(
            _make_trace_frame(
                step=len(frames),
                message=message,
                problem=problem,
                current=current,
                frontier_nodes=frontier_snapshot(),
                expanded=expanded,
                reached=reached,
                solution=solution,
                score_label="depth",
            )
        )

    add_frame(f"Initialize the frontier with {problem.initial}.")

    while frontier:
        node = frontier.popleft() if is_bfs else frontier.pop()
        add_frame(f"Select {node.state} from the frontier.", current=node)

        if problem.goal_test(node.state):
            add_frame(f"Goal found at {node.state}.", current=node, solution=node)
            return {"title": title, "frames": frames}

        if is_bfs:
            expanded.append(node.state)
            for child in node.expand(problem):
                if child.state in reached:
                    continue
                reached.add(child.state)
                if problem.goal_test(child.state):
                    add_frame(
                        f"Discover goal {child.state} from {node.state}.",
                        current=child,
                        solution=child,
                    )
                    return {"title": title, "frames": frames}
                frontier.append(child)
            add_frame(f"Add unvisited neighbors of {node.state} to the frontier.", current=node)
        else:
            if node.state in reached:
                add_frame(f"Skip {node.state}; it was already reached.", current=node)
                continue
            if node.depth >= limit:
                add_frame(f"Depth limit reached at {node.state}.", current=node)
                continue
            reached.add(node.state)
            expanded.append(node.state)
            children = [child for child in node.expand(problem) if child.state not in reached]
            frontier.extend(reversed(children))
            add_frame(f"Push unvisited neighbors of {node.state} onto the stack.", current=node)

    add_frame("The frontier is empty; no solution was found.")
    return {"title": title, "frames": frames}


def _trace_best_first_search(problem, title, score_fn, score_label):
    frontier = []
    counter = itertools.count()
    start = Node(problem.initial)
    heapq.heappush(frontier, (score_fn(start), next(counter), start))
    best_score = {start.state: score_fn(start)}
    expanded = []
    reached = {problem.initial}
    frames = []

    def frontier_snapshot():
        return [item[2] for item in sorted(frontier, key=lambda item: (item[0], item[1]))]

    def add_frame(message, current=None, solution=None):
        frames.append(
            _make_trace_frame(
                step=len(frames),
                message=message,
                problem=problem,
                current=current,
                frontier_nodes=frontier_snapshot(),
                expanded=expanded,
                reached=reached,
                solution=solution,
                score_label=score_label,
                score_fn=score_fn,
            )
        )

    add_frame(f"Initialize the priority queue with {problem.initial}.")

    while frontier:
        _, _, node = heapq.heappop(frontier)
        add_frame(f"Select {node.state} with lowest {score_label}.", current=node)

        if problem.goal_test(node.state):
            add_frame(f"Goal found at {node.state}.", current=node, solution=node)
            return {"title": title, "frames": frames}

        expanded.append(node.state)
        for child in node.expand(problem):
            score = score_fn(child)
            if child.state not in best_score or score < best_score[child.state]:
                best_score[child.state] = score
                reached.add(child.state)
                heapq.heappush(frontier, (score, next(counter), child))
        add_frame(f"Add or improve neighbors of {node.state} in the priority queue.", current=node)

    add_frame("The frontier is empty; no solution was found.")
    return {"title": title, "frames": frames}


def trace_romania_search(problem, algorithm_name):
    if algorithm_name == "Breadth-first graph search":
        return _trace_queue_search(problem, mode="bfs")
    if algorithm_name == "Depth-first graph search":
        return _trace_queue_search(problem, mode="dfs")
    if algorithm_name == "Uniform-cost search":
        return _trace_best_first_search(problem, algorithm_name, lambda node: node.path_cost, "g(n)")
    if algorithm_name == "Greedy best-first search":
        return _trace_best_first_search(problem, algorithm_name, problem.h, "h(n)")
    if algorithm_name == "A* search":
        return _trace_best_first_search(problem, algorithm_name, lambda node: node.path_cost + problem.h(node), "f(n)")
    raise ValueError(f"Unknown algorithm: {algorithm_name}")


def show_romania_search_animation(start="Arad", goal="Bucharest", interval_ms=900):
    heuristic = romania_sld_to_bucharest if goal == "Bucharest" else {}
    algorithm_names = [
        "Breadth-first graph search",
        "Depth-first graph search",
        "Uniform-cost search",
        "Greedy best-first search",
        "A* search",
    ]
    traces = {}
    for name in algorithm_names:
        problem = GraphProblem(start, goal, romania_map, heuristic)
        traces[name] = trace_romania_search(problem, name)

    payload = {
        "start": start,
        "goal": goal,
        "interval": interval_ms,
        "positions": romania_locations,
        "labels": romania_label_offsets,
        "edges": _romania_edges_for_display(romania_map),
        "traces": traces,
    }
    container_id = f"romania-search-{uuid.uuid4().hex}"

    return HTML(f'''
<div id="{container_id}" class="romania-search-viz">
  <style>
    #{container_id} {{
      --ink: #1f2937;
      --muted: #6b7280;
      --line: #9ca3af;
      --panel: #ffffff;
      --border: #d1d5db;
      --frontier: #93c5fd;
      --expanded: #fca5a5;
      --current: #fbbf24;
      --solution: #86efac;
      color: var(--ink);
      font-family: system-ui, -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
      max-width: 980px;
    }}
    #{container_id} .viz-shell {{
      background: var(--panel);
      border: 1px solid var(--border);
      border-radius: 8px;
      padding: 14px;
    }}
    #{container_id} .toolbar {{
      align-items: center;
      display: grid;
      gap: 10px;
      grid-template-columns: minmax(210px, 1fr) auto auto auto minmax(160px, 1fr) auto;
      margin-bottom: 10px;
    }}
    #{container_id} select,
    #{container_id} button,
    #{container_id} input[type="range"] {{
      font: inherit;
    }}
    #{container_id} select,
    #{container_id} button {{
      background: #f9fafb;
      border: 1px solid var(--border);
      border-radius: 6px;
      color: var(--ink);
      min-height: 34px;
      padding: 6px 10px;
    }}
    #{container_id} button {{ cursor: pointer; }}
    #{container_id} button:hover,
    #{container_id} select:hover {{ border-color: #64748b; }}
    #{container_id} .counter {{
      color: var(--muted);
      font-size: 13px;
      text-align: right;
      white-space: nowrap;
    }}
    #{container_id} svg {{
      background: #f8fafc;
      border: 1px solid #e5e7eb;
      border-radius: 8px;
      display: block;
      height: auto;
      width: 100%;
    }}
    #{container_id} .road {{
      stroke: var(--line);
      stroke-linecap: round;
      stroke-width: 3;
    }}
    #{container_id} .road.active {{
      stroke: #2563eb;
      stroke-width: 6;
    }}
    #{container_id} .road.solution {{
      stroke: #16a34a;
      stroke-width: 7;
    }}
    #{container_id} .distance {{
      fill: #475569;
      font-size: 10px;
      paint-order: stroke;
      stroke: #f8fafc;
      stroke-width: 3px;
    }}
    #{container_id} .city circle {{
      fill: #ffffff;
      stroke: #64748b;
      stroke-width: 2;
    }}
    #{container_id} .city text {{
      fill: var(--ink);
      font-size: 13px;
      paint-order: stroke;
      stroke: #f8fafc;
      stroke-width: 4px;
    }}
    #{container_id} .city.expanded circle {{ fill: var(--expanded); }}
    #{container_id} .city.frontier circle {{ fill: var(--frontier); }}
    #{container_id} .city.current circle {{
      fill: var(--current);
      stroke: #92400e;
      stroke-width: 4;
    }}
    #{container_id} .city.solution circle {{
      fill: var(--solution);
      stroke: #15803d;
      stroke-width: 4;
    }}
    #{container_id} .city.start text,
    #{container_id} .city.goal text {{ font-weight: 700; }}
    #{container_id} .city.goal circle {{ stroke-dasharray: 5 3; }}
    #{container_id} .status {{
      font-size: 15px;
      margin-top: 10px;
      min-height: 24px;
    }}
    #{container_id} .legend {{
      display: flex;
      flex-wrap: wrap;
      gap: 8px 14px;
      margin-top: 8px;
    }}
    #{container_id} .legend span {{
      align-items: center;
      color: var(--muted);
      display: inline-flex;
      font-size: 13px;
      gap: 6px;
    }}
    #{container_id} .swatch {{
      border: 1px solid #64748b;
      border-radius: 999px;
      display: inline-block;
      height: 12px;
      width: 12px;
    }}
    #{container_id} .details {{
      display: grid;
      gap: 12px;
      grid-template-columns: minmax(0, 1fr) minmax(0, 1.25fr);
      margin-top: 12px;
    }}
    #{container_id} .metrics,
    #{container_id} .frontier-box {{
      background: #f9fafb;
      border: 1px solid #e5e7eb;
      border-radius: 8px;
      font-size: 13px;
      line-height: 1.45;
      min-height: 74px;
      padding: 10px;
    }}
    #{container_id} .frontier-box ol {{
      margin: 6px 0 0 18px;
      padding: 0;
    }}
    @media (max-width: 720px) {{
      #{container_id} .toolbar {{ grid-template-columns: 1fr 1fr 1fr; }}
      #{container_id} .toolbar select,
      #{container_id} .toolbar input[type="range"] {{ grid-column: 1 / -1; }}
      #{container_id} .counter {{ text-align: left; }}
      #{container_id} .details {{ grid-template-columns: 1fr; }}
    }}
  </style>
  <div class="viz-shell">
    <div class="toolbar">
      <select class="algorithm" aria-label="Algorithm"></select>
      <button type="button" class="play">Play</button>
      <button type="button" class="prev">Prev</button>
      <button type="button" class="next">Next</button>
      <input class="step" type="range" min="0" value="0" />
      <span class="counter"></span>
    </div>
    <svg class="map" viewBox="0 0 680 500" role="img" aria-label="Romania map search animation"></svg>
    <div class="status"></div>
    <div class="legend">
      <span><i class="swatch" style="background: var(--frontier)"></i>Frontier</span>
      <span><i class="swatch" style="background: var(--expanded)"></i>Expanded</span>
      <span><i class="swatch" style="background: var(--current)"></i>Current</span>
      <span><i class="swatch" style="background: var(--solution)"></i>Solution path</span>
    </div>
    <div class="details">
      <div class="metrics"></div>
      <div class="frontier-box"></div>
    </div>
  </div>
  <script>
    (() => {{
      const payload = {json.dumps(payload)};
      const root = document.getElementById({json.dumps(container_id)});
      const svg = root.querySelector('svg.map');
      const select = root.querySelector('select.algorithm');
      const play = root.querySelector('button.play');
      const prev = root.querySelector('button.prev');
      const next = root.querySelector('button.next');
      const slider = root.querySelector('input.step');
      const counter = root.querySelector('.counter');
      const status = root.querySelector('.status');
      const metrics = root.querySelector('.metrics');
      const frontierBox = root.querySelector('.frontier-box');
      const ns = 'http://www.w3.org/2000/svg';
      let algorithm = Object.keys(payload.traces)[0];
      let frames = payload.traces[algorithm].frames;
      let step = 0;
      let timer = null;

      function makeSvg(name, attrs = {{}}) {{
        const element = document.createElementNS(ns, name);
        Object.entries(attrs).forEach(([key, value]) => element.setAttribute(key, value));
        return element;
      }}

      function fmt(value) {{
        if (value === null || value === undefined) return '';
        return Number.isInteger(value) ? String(value) : Number(value).toFixed(1);
      }}

      function escapeText(value) {{
        return String(value).replace(/[&<>"']/g, (ch) => ({{
          '&': '&amp;', '<': '&lt;', '>': '&gt;', '"': '&quot;', "'": '&#039;'
        }}[ch]));
      }}

      function edgeKey(a, b) {{
        return [a, b].sort().join('|');
      }}

      function pathEdgeSet(path) {{
        const keys = new Set();
        for (let i = 0; i < path.length - 1; i += 1) {{
          keys.add(edgeKey(path[i], path[i + 1]));
        }}
        return keys;
      }}

      function drawBaseMap() {{
        svg.innerHTML = '';
        const roadLayer = makeSvg('g', {{ class: 'roads' }});
        const labelLayer = makeSvg('g', {{ class: 'distance-labels' }});
        const cityLayer = makeSvg('g', {{ class: 'cities' }});

        payload.edges.forEach((edge) => {{
          const [x1, y1] = payload.positions[edge.from];
          const [x2, y2] = payload.positions[edge.to];
          const key = edgeKey(edge.from, edge.to);
          roadLayer.appendChild(makeSvg('line', {{
            class: 'road',
            'data-key': key,
            x1, y1, x2, y2,
          }}));
          const text = makeSvg('text', {{
            class: 'distance',
            x: (x1 + x2) / 2,
            y: (y1 + y2) / 2 - 4,
            'text-anchor': 'middle',
          }});
          text.textContent = edge.distance;
          labelLayer.appendChild(text);
        }});

        Object.entries(payload.positions).forEach(([city, coords]) => {{
          const [x, y] = coords;
          const group = makeSvg('g', {{ class: 'city', 'data-city': city }});
          group.appendChild(makeSvg('circle', {{ cx: x, cy: y, r: 9 }}));
          const [dx, dy, anchor] = payload.labels[city] || [12, -10, 'start'];
          const label = makeSvg('text', {{
            x: x + dx,
            y: y + dy,
            'text-anchor': anchor,
          }});
          label.textContent = city;
          group.appendChild(label);
          cityLayer.appendChild(group);
        }});

        svg.appendChild(roadLayer);
        svg.appendChild(labelLayer);
        svg.appendChild(cityLayer);
      }}

      function render() {{
        const frame = frames[step] || {{ frontier: [], expanded: [], path: [] }};
        const frontierSet = new Set((frame.frontier || []).map((item) => item.state));
        const expandedSet = new Set(frame.expanded || []);
        const path = (frame.solution_path && frame.solution_path.length) ? frame.solution_path : (frame.path || []);
        const activeEdges = pathEdgeSet(path);
        const solutionMode = Boolean(frame.solution_path && frame.solution_path.length);

        root.querySelectorAll('.road').forEach((road) => {{
          road.setAttribute('class', activeEdges.has(road.dataset.key)
            ? (solutionMode ? 'road solution' : 'road active')
            : 'road');
        }});

        root.querySelectorAll('.city').forEach((cityGroup) => {{
          const city = cityGroup.dataset.city;
          const classes = ['city'];
          if (city === payload.start) classes.push('start');
          if (city === payload.goal) classes.push('goal');
          if (expandedSet.has(city)) classes.push('expanded');
          if (frontierSet.has(city)) classes.push('frontier');
          if (city === frame.current) classes.push('current');
          if (path.includes(city) && solutionMode) classes.push('solution');
          cityGroup.setAttribute('class', classes.join(' '));
        }});

        const maxStep = Math.max(frames.length - 1, 0);
        slider.max = String(maxStep);
        slider.value = String(step);
        counter.textContent = `${{step + 1}} / ${{frames.length}}`;
        status.textContent = frame.message || '';

        const pathText = path.length ? path.join(' -> ') : 'None yet';
        const solutionCost = frame.solution_cost === null || frame.solution_cost === undefined
          ? 'None yet'
          : fmt(frame.solution_cost);
        metrics.innerHTML = `
          <strong>${{escapeText(payload.traces[algorithm].title)}}</strong><br>
          Current: ${{escapeText(frame.current || 'None')}}<br>
          Current path: ${{escapeText(pathText)}}<br>
          Expanded states: ${{(frame.expanded || []).length}}<br>
          Solution cost: ${{escapeText(solutionCost)}}
        `;

        if (!frame.frontier || frame.frontier.length === 0) {{
          frontierBox.innerHTML = '<strong>Frontier</strong><br>Empty';
        }} else {{
          const items = frame.frontier.map((entry) => {{
            const parts = [`g=${{fmt(entry.g)}}`, `depth=${{fmt(entry.depth)}}`];
            if (entry.h !== null && entry.h !== undefined) parts.push(`h=${{fmt(entry.h)}}`);
            if (entry.score !== null && entry.score !== undefined && frame.score_label) {{
              parts.push(`${{frame.score_label}}=${{fmt(entry.score)}}`);
            }}
            return `<li>${{escapeText(entry.state)}} <span style="color:#6b7280">(${{escapeText(parts.join(', '))}})</span></li>`;
          }}).join('');
          frontierBox.innerHTML = `<strong>Frontier</strong><ol>${{items}}</ol>`;
        }}
      }}

      function stop() {{
        if (timer) window.clearInterval(timer);
        timer = null;
        play.textContent = 'Play';
      }}

      function start() {{
        stop();
        play.textContent = 'Pause';
        timer = window.setInterval(() => {{
          if (step >= frames.length - 1) {{
            stop();
            return;
          }}
          step += 1;
          render();
        }}, payload.interval);
      }}

      Object.keys(payload.traces).forEach((name) => {{
        const option = document.createElement('option');
        option.value = name;
        option.textContent = name;
        select.appendChild(option);
      }});

      select.addEventListener('change', () => {{
        stop();
        algorithm = select.value;
        frames = payload.traces[algorithm].frames;
        step = 0;
        render();
      }});
      play.addEventListener('click', () => timer ? stop() : start());
      prev.addEventListener('click', () => {{ stop(); step = Math.max(0, step - 1); render(); }});
      next.addEventListener('click', () => {{ stop(); step = Math.min(frames.length - 1, step + 1); render(); }});
      slider.addEventListener('input', () => {{ stop(); step = Number(slider.value); render(); }});

      drawBaseMap();
      render();
    }})();
  </script>
</div>
''')


In [ ]:
show_romania_search_animation()


## Demo: Simple problem-solving agent


In [ ]:
def romania_problem_factory(state, goal):
    return GraphProblem(state, goal, romania_map, romania_sld_to_bucharest)

agent = SimpleProblemSolvingAgent(
    initial_state="Arad",
    goal="Bucharest",
    problem_factory=romania_problem_factory,
    search_algorithm=astar_search,
)

current_city = "Arad"
for step in range(5):
    action = agent(current_city)
    print(f'Step {step + 1}: at {current_city}, action = {action}')
    if action is None:
        break
    current_city = action
